In [1]:
from Load_Data import load_ptb_xl_data
import torch
from VQ_VAE import Encoder_net, Quantize_net
from NanoGPT import GPTLanguageModel
from torch import optim
from Train_Transformer import training


path = "../Data/"
train_data, test_data, val_data, train_id, test_id, val_id, train_leads, test_leads, val_leads = load_ptb_xl_data(path)

# Define parameters
batch_size = 256 # how many independent sequences will we process in parallel?
learning_rate = 3e-4
device = torch.device(0)
block_size = 306 # what is the maximum context length for predictions?
eval_interval = 10
n_embd = 384
num_classes = 12
n_head = 8
n_layer = 8
dropout = 0.2

encoder = Encoder_net(4)
vocab_size = 64
size = 10

n_head = size
n_layer = size
print("Vocab size: ", vocab_size)

encoder = Encoder_net(4)
temp = torch.randn(1, 1, 5000)  # Dummy input to initialize the quantizer
temp = encoder(temp)  # Forward pass to initialize weights
encoding_size = temp.shape[1]  # Get the output size of the encoder
quantizer = Quantize_net(encoding_size, vocab_size)

encoder.load_state_dict(torch.load("../Models/VQ-VAE/Encoder_vocab_size_"+str(vocab_size)+"_4_epoch20.pth", weights_only=False))
quantizer.load_state_dict(torch.load("../Models/VQ-VAE/Quantizer_vocab_size_"+str(vocab_size)+"_4_epoch20.pth", weights_only=False))

model = GPTLanguageModel(block_size=block_size, vocab_size=vocab_size, n_embd=n_embd, n_layer=n_layer, n_head=n_head, dropout=dropout, num_classes=num_classes)
optimizer = optim.AdamW(model.parameters(), lr=learning_rate)


training(model, encoder, quantizer, train_data, test_data, optimizer, device, vocab_size, size, 50, batch_size)

/pools/apollon/ummisco/data/projects/deepecg/analyses/17.gan/ECGenerator/GitHub/Scr/Load_Data.py:23: RuntimeWarning: invalid value encountered in divide
  return(-1+((Z-mini)*(2))/(maxi-mini))


(208904, 1, 5000)


Vocab size:  64
Epoch 1/50, Train Loss: 1.4020, Train Acc: 0.5762, Test Loss: 1.1799, Test Acc: 0.6143


KeyboardInterrupt: 